# RF Signal Classifier â€” 2D Training (ResNet-18 + STFT-RADN)
**Run on GPU: Runtime â†’ Change runtime type â†’ T4 GPU**

### Steps
1. Run Cell 1 â€” check GPU
2. Run Cell 2 â€” upload your 6 `.npy` data files
3. Run remaining cells in order

In [ ]:
# â”€â”€ Cell 1: GPU Check â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── Cell 2: Mount Google Drive & Copy Data ────────────────────
# Since the 6 .npy files were uploaded during a previous run,
# we will just mount your Google Drive and copy them directly.
import os
import shutil
from google.colab import drive

drive.mount('/content/drive')

os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/checkpoints', exist_ok=True)

# UPDATE THIS PATH if your data is saved in a different folder in your Drive
drive_data_path = '/content/drive/MyDrive/rf_data/rf_data' 

files_to_copy = ['spectrograms_all.npy', 'y_all.npy', 'snrs_all.npy', 'train_idx.npy', 'val_idx.npy', 'test_idx.npy']
print('Copying files from Drive to local Colab storage for speed...')
for fname in files_to_copy:
    src = os.path.join(drive_data_path, fname)
    dst = os.path.join('/content/data', fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'  Copied: {fname} ({size_mb:.1f} MB)')
    else:
        print(f'  [ERROR] Could not find {fname} in {drive_data_path}')

print('\nFiles ready in /content/data:')
for f in sorted(os.listdir('/content/data')):
    print(f'  {f}')

In [ ]:
# â”€â”€ Cell 3: Model Definitions (ResNet18_2D + STFT-RADN) â”€â”€â”€â”€â”€â”€â”€
import torch
import torch.nn as nn
import torch.nn.functional as F

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.dropout = nn.Dropout2d(p=0.3)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * self.expansion),
            )
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class ResNet18_2D(nn.Module):
    def __init__(self, num_classes=11, in_channels=1):
        super().__init__()
        self.in_planes = 16
        self.conv1 = nn.Conv2d(in_channels, 16, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.layer1 = self._make_layer(16,  2, stride=1)
        self.layer2 = self._make_layer(32, 2, stride=2)
        self.layer3 = self._make_layer(64, 2, stride=2)
        self.layer4 = self._make_layer(128, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None: nn.init.constant_(m.bias, 0)
    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        x = self.avgpool(x)
        return self.fc(torch.flatten(x, 1))

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.mlp = nn.Sequential(nn.Linear(channels, mid, bias=False), nn.ReLU(inplace=True), nn.Linear(mid, channels, bias=False))
    def forward(self, x):
        scale = torch.sigmoid(self.mlp(x.mean(dim=[2,3])) + self.mlp(x.amax(dim=[2,3])))
        return x * scale.unsqueeze(-1).unsqueeze(-1)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
    def forward(self, x):
        scale = torch.sigmoid(self.conv(torch.cat([x.mean(1,keepdim=True), x.amax(1,keepdim=True)], dim=1)))
        return x * scale

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention(spatial_kernel)
    def forward(self, x):
        return self.spatial_att(self.channel_att(x))

class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate):
        super().__init__()
        self.layer = nn.Sequential(nn.BatchNorm2d(in_channels), nn.ReLU(inplace=True), nn.Conv2d(in_channels, growth_rate, 3, padding=1, bias=False))
    def forward(self, x):
        return torch.cat([x, self.layer(x)], dim=1)

class ResidualDenseBlock(nn.Module):
    def __init__(self, in_channels=64, growth_rate=32, num_layers=4):
        super().__init__()
        layers, ch = [], in_channels
        for _ in range(num_layers):
            layers.append(DenseLayer(ch, growth_rate)); ch += growth_rate
        self.dense_layers = nn.Sequential(*layers)
        self.lff = nn.Sequential(nn.Conv2d(ch, in_channels, 1, bias=False), nn.BatchNorm2d(in_channels))
        self.cbam = CBAM(in_channels)
    def forward(self, x):
        return self.cbam(self.lff(self.dense_layers(x))) + x

class STFTRADN(nn.Module):
    def __init__(self, num_classes=11, in_channels=1, base_channels=64, growth_rate=32, num_dense_layers=4, num_rdb=3):
        super().__init__()
        self.num_rdb = num_rdb
        self.sfe1 = nn.Conv2d(in_channels, base_channels, 3, padding=1, bias=False)
        self.sfe2 = nn.Sequential(nn.BatchNorm2d(base_channels), nn.ReLU(inplace=True), nn.Conv2d(base_channels, base_channels, 3, padding=1, bias=False))
        self.rdbs = nn.ModuleList([ResidualDenseBlock(base_channels, growth_rate, num_dense_layers) for _ in range(num_rdb)])
        self.gff = nn.Sequential(nn.Conv2d(base_channels * num_rdb, base_channels, 1, bias=False), nn.BatchNorm2d(base_channels), nn.ReLU(inplace=True))
        self.backbone = nn.Sequential(
            nn.Conv2d(base_channels, 128, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)))
        self.fc = nn.Linear(256, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d): nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None: nn.init.constant_(m.bias, 0)
    def forward(self, x):
        f0 = self.sfe1(x); f1 = self.sfe2(f0)
        rdb_out, h = [], f1
        for rdb in self.rdbs:
            h = rdb(h); rdb_out.append(h)
        out = self.gff(torch.cat(rdb_out, dim=1)) + f0
        return self.fc(torch.flatten(self.backbone(out), 1))

print('[OK] Models defined: ResNet18_2D, STFTRADN')

In [ ]:
# â”€â”€ Cell 4: Dataset & DataLoader â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import numpy as np
from torch.utils.data import Dataset, DataLoader

class SpectrogramDataset(Dataset):
    """Loads pre-computed STFT spectrograms via memory-mapping."""
    def __init__(self, data_dir='/content/data', split='train', in_channels=1):
        self.split = split
        self.in_channels = in_channels
        self._spectrograms = np.load(f'{data_dir}/spectrograms_all.npy', mmap_mode='r')
        self._labels = np.load(f'{data_dir}/y_all.npy')
        self._snrs   = np.load(f'{data_dir}/snrs_all.npy')
        self._indices = np.load(f'{data_dir}/{split}_idx.npy')
        import torchvision.transforms as T
        self.transform = T.RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), value=0) if split == 'train' else None
    def __len__(self):
        return len(self._indices)
    def __getitem__(self, idx):
        i = self._indices[idx]
        spec = np.array(self._spectrograms[i], dtype=np.float32)
        if self.in_channels == 1:
            spec = spec[:1]
        signal_tensor = torch.from_numpy(spec)
        if self.transform is not None:
            signal_tensor = self.transform(signal_tensor)
        return signal_tensor, torch.tensor(int(self._labels[i]), dtype=torch.long), self._snrs[i]

def get_loader(split, batch_size, in_channels, num_workers=2, data_dir='/content/data'):
    ds = SpectrogramDataset(data_dir, split, in_channels)
    return DataLoader(ds, batch_size=batch_size, shuffle=(split=='train'),
                      num_workers=num_workers, pin_memory=True, persistent_workers=(num_workers>0))

print('[OK] Dataset classes defined')

In [ ]:
# â”€â”€ Cell 5: Training Config â€” EDIT HERE â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
CONFIG = dict(
    model        = 'resnet18',   # 'resnet18' or 'stft-radn'
    input_mode   = 'grayscale',  # 'grayscale' (1ch) or 'hybrid' (3ch)
    batch_size   = 256,
    epochs       = 30,
    lr           = 1e-3,
    weight_decay = 1e-2,
    patience     = 10,           # early stopping
    num_workers  = 2,
    data_dir     = '/content/data',
    save_dir     = '/content/checkpoints',
)
print('Config:', CONFIG)

In [ ]:
# â”€â”€ Cell 6: Build Model + Loaders â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import os, time
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
in_ch  = 1 if CONFIG['input_mode'] == 'grayscale' else 3
os.makedirs(CONFIG['save_dir'], exist_ok=True)

# Loaders
kw = dict(batch_size=CONFIG['batch_size'], in_channels=in_ch,
          num_workers=CONFIG['num_workers'], data_dir=CONFIG['data_dir'])
train_loader = get_loader('train', **kw)
val_loader   = get_loader('val',   **kw)
test_loader  = get_loader('test',  **kw)

num_classes = len(np.unique(np.load(CONFIG['data_dir'] + '/y_all.npy')))

# Model
if CONFIG['model'] == 'resnet18':
    model = ResNet18_2D(num_classes=num_classes, in_channels=in_ch)
else:
    model = STFTRADN(num_classes=num_classes, in_channels=in_ch)
model = model.to(device)

optimizer = AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
criterion = torch.nn.CrossEntropyLoss()
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

total_params = sum(p.numel() for p in model.parameters())
print(f'Device     : {device}')
print(f'Model      : {CONFIG["model"]}  ({total_params:,} params)')
print(f'Train      : {len(train_loader.dataset):,} samples  ({len(train_loader)} batches)')
print(f'Val        : {len(val_loader.dataset):,} samples')
print(f'Test       : {len(test_loader.dataset):,} samples')
print(f'Classes    : {num_classes}')

In [ ]:
# â”€â”€ Cell 7: Training Loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
best_val_acc = 0.0
best_path    = f"{CONFIG['save_dir']}/best_{CONFIG['model']}.pt"
patience_ctr = 0
best_val_loss = float('inf')

print('='*74)
print(f'{"Ep":>4} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>9} | {"Val Acc":>8} | {"LR":>8} | {"Time":>6}')
print('-'*74)

for epoch in range(1, CONFIG['epochs'] + 1):
    t0 = time.time()

    # Train
    model.train()
    tr_loss, tr_corr, tr_total = 0.0, 0, 0
    for signals, labels, _ in train_loader:
        signals, labels = signals.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(signals)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        tr_loss  += loss.item() * signals.size(0)
        tr_corr  += out.argmax(1).eq(labels).sum().item()
        tr_total += labels.size(0)

    # Validate
    model.eval()
    vl_loss, vl_corr, vl_total = 0.0, 0, 0
    with torch.no_grad():
        for signals, labels, _ in val_loader:
            signals, labels = signals.to(device), labels.to(device)
            out  = model(signals)
            loss = criterion(out, labels)
            vl_loss  += loss.item() * signals.size(0)
            vl_corr  += out.argmax(1).eq(labels).sum().item()
            vl_total += labels.size(0)

    tr_loss /= tr_total; tr_acc = 100. * tr_corr / tr_total
    vl_loss /= vl_total; vl_acc = 100. * vl_corr / vl_total
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(vl_loss)
    elapsed = time.time() - t0

    marker = ' *' if vl_acc > best_val_acc else ''
    print(f'{epoch:4d} | {tr_loss:10.4f} | {tr_acc:8.2f}% | {vl_loss:9.4f} | {vl_acc:7.2f}% | {current_lr:8.6f} | {elapsed:5.1f}s{marker}')

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_acc': vl_acc, 'val_loss': vl_loss,
                    'model_name': CONFIG['model'], 'num_classes': num_classes,
                    'in_channels': in_ch}, best_path)

    if vl_loss < best_val_loss - 1e-4:
        best_val_loss = vl_loss; patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= CONFIG['patience']:
            print(f'\n[EARLY STOP] No val loss improvement for {CONFIG["patience"]} epochs.')
            break

print('='*74)
print(f'\n[DONE] Best val acc: {best_val_acc:.2f}%  |  Checkpoint: {best_path}')

In [ ]:
# â”€â”€ Cell 8: Final Test Evaluation + Per-SNR Accuracy â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
checkpoint = torch.load(best_path, map_location=device, weights_only=True)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

te_loss, te_corr, te_total = 0.0, 0, 0
snr_correct, snr_total = {}, {}

with torch.no_grad():
    for signals, labels, snrs in test_loader:
        signals, labels = signals.to(device), labels.to(device)
        out  = model(signals)
        loss = criterion(out, labels)
        pred = out.argmax(1)
        te_loss  += loss.item() * signals.size(0)
        te_corr  += pred.eq(labels).sum().item()
        te_total += labels.size(0)
        for i in range(labels.size(0)):
            snr = int(snrs[i].item())
            snr_correct.setdefault(snr, 0)
            snr_total.setdefault(snr, 0)
            snr_total[snr] += 1
            if pred[i] == labels[i]: snr_correct[snr] += 1

te_loss /= te_total
te_acc   = 100. * te_corr / te_total
print(f'\n[TEST RESULT]  Loss: {te_loss:.4f}  |  Accuracy: {te_acc:.2f}%')

print('\n--- Per-SNR Accuracy ---')
print(f'{"SNR (dB)":>10} | {"Accuracy":>10} | Bar')
print('-'*50)
snr_accs = []
for snr in sorted(snr_correct):
    acc = 100. * snr_correct[snr] / snr_total[snr]
    snr_accs.append(acc)
    bar = '#' * int(acc / 5)
    print(f'{snr:>8d} dB | {acc:9.2f}% | {bar}')
print('-'*50)
print(f'{"Average":>10} | {np.mean(snr_accs):9.2f}%')

In [ ]:
# â”€â”€ Cell 9: Download Best Checkpoint â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from google.colab import files
print(f'Downloading {best_path} ...')
files.download(best_path)
print('Done!')